# Bước 2 — Chốt nguồn, URL và khả năng crawl VnExpress

**Trạng thái:** VnExpress **vẫn được giữ** trong bộ 5 nguồn.

`categories-sitemap.xml` trả về trang chủ chỉ có nghĩa là **một endpoint sitemap không dùng được**, không phải loại bỏ nguồn. VnExpress đã được xác minh lại bằng:

1. Sitemap index và các sitemap con.
2. 8 RSS chuyên mục.
3. 8 chuyên mục × 5 trang danh mục.
4. 8 bài viết mẫu, mỗi lĩnh vực một bài.

**Thời điểm khảo sát:** 20/11/2026, UTC+7.

In [1]:
from pathlib import Path
import subprocess, sys
import pandas as pd

RUN_LIVE = False
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = PROJECT_ROOT / "outputs"

if RUN_LIVE:
    for script in [
        "research/vnexpress_sitemap_probe.py",
        "research/vnexpress_category_harvest.py",
        "research/vnexpress_article_probe.py",
        "research/build_step2_outputs.py",
    ]:
        subprocess.run([sys.executable, str(PROJECT_ROOT / script)], check=True, cwd=PROJECT_ROOT)

matrix = pd.read_csv(OUT / "step2_vnexpress_matrix.csv", encoding="utf-8-sig")
categories = pd.read_csv(OUT / "step2_vnexpress_categories.csv", encoding="utf-8-sig")
sitemaps = pd.read_csv(OUT / "step2_vnexpress_sitemaps.csv", encoding="utf-8-sig")
articles = pd.read_csv(OUT / "step2_vnexpress_articles.csv", encoding="utf-8-sig")
plan = pd.read_csv(OUT / "step2_source_plan.csv", encoding="utf-8-sig")
print("Đã đọc 5 bảng Bước 2")

Đã đọc 5 bảng Bước 2


## 1. Kết quả 8 chuyên mục × 5 trang

Cả **40/40 trang** trả HTTP 200. Số URL bài viết là **số tối thiểu** theo regex; các URL dạng `-tong-thuat` được bổ sung từ RSS.

In [2]:
page_summary = (categories.groupby("Label", sort=False)
    .agg(Pages=("Page", "nunique"),
         HTTP_200=("HTTP", lambda s: int((s == 200).sum())),
         Unique_URLs=("Cumulative_Unique", "max"),
         Min_Links_Page=("Links_On_Page", "min"),
         Max_Links_Page=("Links_On_Page", "max"))
    .reset_index())
assert len(page_summary) == 8
assert (page_summary["HTTP_200"] == 5).all()
assert (page_summary["Unique_URLs"] >= 150).all()
page_summary

,Label,Pages,HTTP_200,Unique_URLs,Min_Links_Page,Max_Links_Page
0,Kinh tế,5,5,169,29,50
1,Thế giới,5,5,166,30,46
2,Thể thao,5,5,163,29,46
3,Pháp luật,5,5,167,30,47
4,Giáo dục,5,5,171,29,52
5,Sức khỏe,5,5,169,29,50
6,Công nghệ,5,5,161,27,48
7,Du lịch,5,5,184,30,64


## 2. RSS và khả năng dự phòng

Mỗi RSS VnExpress có 60 mục. Với 7/8 chuyên mục, gần như toàn bộ bài RSS đã nằm trong 5 trang danh mục. Công nghệ có 13 bài RSS không trùng do feed `Khoa học công nghệ` rộng hơn trang Công nghệ.

In [3]:
matrix[[
    "Label", "Unique_Category_URLs", "RSS_Items", "RSS_Overlap",
    "Union_Available", "Candidate_Quota", "Quota_Coverage_Pct"
]]

,Label,Unique_Category_URLs,RSS_Items,RSS_Overlap,Union_Available,Candidate_Quota,Quota_Coverage_Pct
0,Kinh tế,169,60,59,170,150,100
1,Thế giới,166,60,59,167,150,100
2,Thể thao,163,60,59,164,150,100
3,Pháp luật,167,60,60,167,150,100
4,Giáo dục,171,60,60,171,150,100
5,Sức khỏe,169,60,60,169,150,100
6,Công nghệ,161,60,47,174,150,100
7,Du lịch,184,60,60,184,150,100


## 3. Sitemap: chỉ index hoạt động, sitemap con trả trang chủ

Sitemap index trả XML hợp lệ với 9.468 mục. Tuy nhiên, tất cả sitemap con được kiểm tra (bài viết theo ngày, Google News, topics, categories) đều trả **HTTP 200 + `text/html` + trang chủ**, không phải XML. Không giả User-Agent Googlebot để lách; sitemap vì vậy **không được dùng làm nguồn chính**.

In [4]:
assert len(sitemaps) == 6
assert (sitemaps["status"] == 200).all()
assert int(sitemaps.loc[sitemaps["name"] == "sitemap_index", "entries"].iloc[0]) == 9468
assert (sitemaps.loc[sitemaps["name"] != "sitemap_index", "is_xml"] == False).all()
sitemaps[["name", "status", "content_type", "is_xml", "entries", "final_url"]]

,name,status,content_type,is_xml,entries,final_url
0,sitemap_index,200,application/xml; charset=utf-8,True,9468.0,https://vnexpress.net/sitemap.xml
1,daily_2026_11_20,200,text/html; charset=utf-8,False,NaN,https://vnexpress.net
2,daily_2026_11_19,200,text/html; charset=utf-8,False,NaN,https://vnexpress.net
3,google_news,200,text/html; charset=utf-8,False,NaN,https://vnexpress.net
4,topics,200,text/html; charset=utf-8,False,NaN,https://vnexpress.net
5,categories,200,text/html; charset=utf-8,False,NaN,https://vnexpress.net


## 4. Khả năng crawl nội dung bài viết

Đã kiểm tra **8 bài, đủ 8 lĩnh vực**: tất cả trả HTTP 200, có tiêu đề, ngày đăng và khối nội dung `.fck_detail`.

- Nội dung: 1.912–6.677 ký tự/bài mẫu.
- `.fck_detail` xuất hiện ở cả 8 bài, kể cả 2 bài tổng thuật không có `article.fck_detail`.
- Ngày đăng lấy được từ JSON-LD `datePublished` ở cả 8 bài.
- Không thấy đăng nhập/paywall/CAPTCHA.
- Sức khỏe có `<h1>` đầu tiên rỗng (logo), phải lấy `h1.title-detail` hoặc JSON-LD headline.

In [5]:
assert len(articles) == 8
assert (articles["HTTP"] == 200).all()
assert (articles["Content_Chars"] >= 1000).all()
assert articles["Content_Selector"].eq(".fck_detail").all()
assert articles["DatePublished"].notna().all()
articles[["Label", "HTTP", "Content_Chars", "Paragraphs", "Images", "DatePublished", "Has_Article_Wrapper"]]

,Label,HTTP,Content_Chars,Paragraphs,Images,DatePublished,Has_Article_Wrapper
0,Kinh tế,200,1924,12,0,2026-11-20T11:11:51+07:00,True
1,Thế giới - tổng thuật,200,3007,12,2,2026-11-19T14:26:53+07:00,False
2,Thể thao - tổng thuật,200,6677,23,5,2026-11-17T18:05:00+07:00,False
3,Pháp luật,200,5697,33,1,2026-11-20T00:00:00+07:00,True
4,Công nghệ,200,2976,10,1,2026-11-20T12:00:41+07:00,True
5,Du lịch,200,3429,17,2,2026-11-20T13:17:41+07:00,True
6,Giáo dục,200,1912,14,2,2026-11-20T12:58:37+07:00,True
7,Sức khỏe,200,3663,15,2,2026-11-20T11:00:00+07:00,True


## 5. Ma trận nguồn cuối cùng

Kế hoạch giữ nguyên **5 nguồn**, **750 URL/lĩnh vực**, tổng **6.000 URL ứng viên**. VnExpress lấy **150 URL/lĩnh vực từ p1–p5**, dự phòng RSS; sitemap con không dùng.

In [6]:
assert len(plan) == 40
assert plan.groupby("Label").size().eq(5).all()
assert plan.groupby("Label")["Candidate_Quota"].sum().eq(750).all()
assert plan["Candidate_Quota"].sum() == 6000
assert (plan.loc[plan["Source"] == "VnExpress", "Primary_Method"] == "category_pages").all()
assert len(plan.loc[plan["Source"] == "VnExpress"]) == 8
plan.groupby("Source")["Candidate_Quota"].sum().sort_values(ascending=False)

Source
VietnamNet    3200
VnExpress     1200
DanTri         800
ThanhNien      400
TuoiTre        400
Name: Candidate_Quota, dtype: int64

## 6. Kết luận Bước 2

1. **VnExpress được giữ**, không bị loại vì một sitemap lỗi.
2. **40/40 trang chuyên mục** và **8/8 bài mẫu** đều truy cập được.
3. VnExpress đủ **150 URL/lĩnh vực** chỉ từ trang chuyên mục.
4. Sitemap con không dùng được với User-Agent minh bạch; RSS và trang chuyên mục đủ thay thế.
5. Kế hoạch cuối cùng: 5 nguồn, 750 URL/lĩnh vực, 6.000 URL ứng viên.
6. **Chưa crawl toàn bộ nội dung**; đó là Bước 3–4.